# 실험 결과 분석 — MAPF-GPT

논문의 주요 실험 결과를 재현하고 시각화합니다.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import numpy as np

matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False

## 1. 논문 Figure 3 재현 — 맵별 Success Rate

In [ ]:
# 논문 Figure 3 데이터 (논문 수치 기반)
data = {
    'Random': {
        'agents':         [8,    16,   32,   64],
        'MAPF-GPT-85M':  [1.00, 1.00, 1.00, 0.98],
        'MAPF-GPT-6M':   [1.00, 1.00, 0.99, 0.95],
        'MAPF-GPT-2M':   [1.00, 1.00, 0.98, 0.90],
        'SCRIMP':         [1.00, 0.99, 0.93, 0.75],
        'DCC':            [1.00, 0.98, 0.85, 0.60],
    },
    'Mazes': {
        'agents':         [8,    16,   32,   64],
        'MAPF-GPT-85M':  [1.00, 0.99, 0.95, 0.80],
        'MAPF-GPT-6M':   [1.00, 0.97, 0.90, 0.72],
        'MAPF-GPT-2M':   [0.99, 0.95, 0.85, 0.62],
        'SCRIMP':         [0.98, 0.88, 0.65, 0.35],
        'DCC':            [0.95, 0.80, 0.50, 0.20],
    },
    'Warehouse': {
        'agents':         [32,   64,   128,  192],
        'MAPF-GPT-85M':  [1.00, 0.99, 0.90, 0.78],
        'MAPF-GPT-6M':   [1.00, 0.98, 0.88, 0.72],
        'MAPF-GPT-2M':   [0.99, 0.95, 0.82, 0.65],
        'SCRIMP':         [1.00, 0.99, 0.94, 0.88],
        'DCC':            [0.98, 0.93, 0.80, 0.62],
    },
    'Cities-tiles': {
        'agents':         [64,   128,  192,  256],
        'MAPF-GPT-85M':  [0.99, 0.95, 0.88, 0.78],
        'MAPF-GPT-6M':   [0.98, 0.92, 0.83, 0.72],
        'MAPF-GPT-2M':   [0.96, 0.88, 0.78, 0.65],
        'SCRIMP':         [0.97, 0.91, 0.82, 0.70],
        'DCC':            [0.94, 0.83, 0.70, 0.55],
    }
}

styles = {
    'MAPF-GPT-85M': ('blue',   '-',  'o',  2.5),
    'MAPF-GPT-6M':  ('blue',   '--', 's',  2.0),
    'MAPF-GPT-2M':  ('blue',   ':',  '^',  2.0),
    'SCRIMP':        ('red',    '-',  'x',  2.0),
    'DCC':           ('green',  '-',  'D',  2.0),
}

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for ax, (map_name, map_data) in zip(axes, data.items()):
    agents = map_data['agents']
    for model, (color, ls, marker, lw) in styles.items():
        ax.plot(agents, map_data[model], color=color, linestyle=ls,
                marker=marker, linewidth=lw, markersize=6, label=model)
    ax.set_title(map_name, fontsize=12, fontweight='bold')
    ax.set_xlabel('에이전트 수', fontsize=10)
    ax.set_ylabel('Success Rate' if map_name == 'Random' else '', fontsize=10)
    ax.set_ylim(0, 1.08)
    ax.grid(alpha=0.25)
    ax.set_xticks(agents)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=5,
           bbox_to_anchor=(0.5, -0.08), fontsize=10)

fig.suptitle('논문 Figure 3 재현 — 맵별 Success Rate 비교', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('figure3_reproduction.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Ablation Study — Table 1 시각화

In [ ]:
# 논문 Table 1 데이터
ablation_data = {
    'Random':      {'6M': 97.6, 'noGoal': 95.7, 'noGA': 97.0, 'noAH': 95.6, 'noC2G': 25.8},
    'Mazes':       {'6M': 74.6, 'noGoal': 71.6, 'noGA': 37.6, 'noAH': 85.8, 'noC2G': 15.1},
    'Warehouse':   {'6M': 94.1, 'noGoal': 92.8, 'noGA': 87.7, 'noAH': 94.8, 'noC2G': 11.5},
    'Cities-tiles':{'6M': 82.0, 'noGoal': 88.4, 'noGA': 79.1, 'noAH': 82.2, 'noC2G': 10.2},
    'Puzzles':     {'6M': 94.0, 'noGoal': 92.7, 'noGA': 92.7, 'noAH': 91.5, 'noC2G': 52.5},
}

scenarios = list(ablation_data.keys())
conditions = ['6M', 'noGoal', 'noGA', 'noAH', 'noC2G']
labels = ['6M (전체)', 'Goal 제거', 'Greedy\nAction 제거', 'Action\nHistory 제거', 'Cost-to-go\n제거']
colors = ['#1976D2', '#66BB6A', '#FFA726', '#AB47BC', '#EF5350']

x = np.arange(len(scenarios))
width = 0.15

fig, ax = plt.subplots(figsize=(14, 6))

for i, (cond, label, color) in enumerate(zip(conditions, labels, colors)):
    values = [ablation_data[s][cond] for s in scenarios]
    offset = (i - 2) * width
    bars = ax.bar(x + offset, values, width, label=label, color=color, alpha=0.85, edgecolor='white')

    # 극단적으로 낮은 값에 강조 표시
    for bar, val in zip(bars, values):
        if val < 30:
            ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                    f'{val:.0f}%', ha='center', va='bottom', fontsize=7,
                    color='red', fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(scenarios, fontsize=11)
ax.set_ylabel('Success Rate (%)', fontsize=12)
ax.set_title('Ablation Study — 입력 정보 제거에 따른 성능 변화 (Table 1 재현)', fontsize=12)
ax.set_ylim(0, 115)
ax.legend(loc='upper right', fontsize=9, ncol=5)
ax.axhline(y=100, color='gray', linestyle='--', alpha=0.3)
ax.grid(axis='y', alpha=0.25)

# 핵심 발견 주석
ax.annotate('Cost-to-go 제거\n→ 치명적 성능 하락',
            xy=(0.15, 26), xytext=(0.5, 45),
            fontsize=9, color='red',
            arrowprops=dict(arrowstyle='->', color='red', lw=1.5))

ax.annotate('Action History 제거\n→ Mazes에서 오히려 향상',
            xy=(1.30, 86), xytext=(2.0, 100),
            fontsize=9, color='purple',
            arrowprops=dict(arrowstyle='->', color='purple', lw=1.5))

plt.tight_layout()
plt.savefig('ablation_study_full.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. 런타임 비교 — Figure 5 재현

In [ ]:
# 논문 Figure 5 데이터 (Warehouse)
num_agents = [32, 64, 96, 128, 160, 192]

runtime = {
    'MAPF-GPT-85M': [0.05, 0.10, 0.20, 0.42, 0.85, 1.55],
    'MAPF-GPT-6M':  [0.01, 0.02, 0.04, 0.07, 0.11, 0.15],
    'MAPF-GPT-2M':  [0.005,0.01, 0.02, 0.04, 0.06, 0.09],
    'DCC':           [0.02, 0.08, 0.22, 0.45, 0.78, 1.20],
    'SCRIMP':        [0.04, 0.15, 0.38, 0.75, 1.25, 1.95],
}

styles_rt = {
    'MAPF-GPT-85M': ('blue',   '-',  'o'),
    'MAPF-GPT-6M':  ('blue',   '--', 's'),
    'MAPF-GPT-2M':  ('blue',   ':',  '^'),
    'DCC':           ('green',  '-',  'D'),
    'SCRIMP':        ('red',    '-',  'x'),
}

fig, ax = plt.subplots(figsize=(8, 5))

for model, (color, ls, marker) in styles_rt.items():
    ax.plot(num_agents, runtime[model], color=color, linestyle=ls,
            marker=marker, linewidth=2, markersize=7, label=model)

ax.set_xlabel('에이전트 수', fontsize=12)
ax.set_ylabel('추론 시간 (초)', fontsize=12)
ax.set_title('논문 Figure 5 재현 — Warehouse 맵 추론 시간 비교', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

# 속도 차이 주석
ax.annotate('192 에이전트 기준:\nMAPF-GPT-2M이\nSCRIMP 대비 13×\nDCC 대비 8× 빠름',
            xy=(192, 0.09), xytext=(140, 1.0),
            fontsize=9,
            arrowprops=dict(arrowstyle='->', color='black', lw=1.5),
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig('runtime_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. LifeLong MAPF 결과 — Table 2 시각화

In [ ]:
# 논문 Table 2 데이터
lmapf_data = {
    'Random':      {'6M': 1.497, '6M tuned': 1.507, 'RHCR': 2.164, 'Follower': 1.637, 'MATS-LP': 1.674},
    'Mazes':       {'6M': 0.908, '6M tuned': 1.087, 'RHCR': 1.554, 'Follower': 1.140, 'MATS-LP': 1.125},
    'Warehouse':   {'6M': 1.113, '6M tuned': 1.270, 'RHCR': 2.352, 'Follower': 2.731, 'MATS-LP': 1.701},
    'Cities-tiles':{'6M': 2.840, '6M tuned': 2.994, 'RHCR': 3.480, 'Follower': 3.271, 'MATS-LP': 3.320},
}

scenarios = list(lmapf_data.keys())
models = ['6M', '6M tuned', 'RHCR', 'Follower', 'MATS-LP']
model_labels = ['MAPF-GPT-6M\n(zero-shot)', 'MAPF-GPT-6M\n(fine-tuned)', 'RHCR', 'Follower', 'MATS-LP']
colors_lmapf = ['#1976D2', '#42A5F5', '#4CAF50', '#FF9800', '#9C27B0']

x = np.arange(len(scenarios))
width = 0.15

fig, ax = plt.subplots(figsize=(12, 6))

for i, (model, label, color) in enumerate(zip(models, model_labels, colors_lmapf)):
    values = [lmapf_data[s][model] for s in scenarios]
    offset = (i - 2) * width
    ax.bar(x + offset, values, width, label=label, color=color, alpha=0.85, edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(scenarios, fontsize=11)
ax.set_ylabel('평균 처리량 (Throughput, 높을수록 좋음)', fontsize=11)
ax.set_title('LifeLong MAPF 성능 비교 (Table 2 재현)', fontsize=12)
ax.legend(loc='upper left', fontsize=9)
ax.grid(axis='y', alpha=0.25)

plt.tight_layout()
plt.savefig('lmapf_throughput.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n핵심 관찰:")
print("- Zero-shot 모델도 Follower, MATS-LP와 경쟁 가능한 성능")
print("- Fine-tuning 후 모든 환경에서 성능 향상")
print("- Warehouse에서는 RHCR, Follower가 여전히 우위")